# LaCAM centralized baseline ( S3) — Colab **or** Kaggle

Runs the **real LaCAM** solver (Kei18/lacam3, via the official
`Cognitive-AI-Systems/pogema-benchmark` integration) on our den520d saturation
densities. CPU-only. **Auto-detects Colab vs Kaggle** (paths + env). 

**Resumable & persisted:** the sweep runs one `(agents, seed)` at a time and writes its
result immediately to `RESULTS` (Drive on Colab; `/kaggle/working` on Kaggle), skipping
anything already done. A disconnect costs at most the episode in flight.

Gates: confirm `liblacam.so` (cell 2) and the smoke test (cell 5) before the sweep.

## 0. Detect platform + paths

In [ ]:
import os
KAGGLE = os.path.exists('/kaggle')
BASE = '/kaggle/working' if KAGGLE else '/content'
BENCH = f'{BASE}/pogema-benchmark'
OURS  = f'{BASE}/mapf-deadlock'
print('platform:', 'Kaggle' if KAGGLE else 'Colab', '| BASE =', BASE)

## 1. System build tools + clone repos

In [ ]:
!apt-get -qq update && apt-get -qq install -y cmake build-essential >/dev/null
if not os.path.exists(BENCH):
    !git clone --recursive -q https://github.com/Cognitive-AI-Systems/pogema-benchmark.git {BENCH}
!cd {BENCH} && git submodule update --init --recursive
if not os.path.exists(OURS):
    !git clone -q --depth 1 https://github.com/tay805/mapf-deadlock.git {OURS}
print('lacam3 present:', os.path.isdir(f'{BENCH}/algorithms/lacam/lacam3'))

## 2. Build `liblacam.so`  *(gate)*

In [ ]:
import glob, shutil
os.chdir(f'{BENCH}/algorithms/lacam')
!cmake -B build -DCMAKE_BUILD_TYPE=Release >/tmp/cmake.log 2>&1 && cmake --build build -j 2 >>/tmp/cmake.log 2>&1; tail -5 /tmp/cmake.log
so = glob.glob(f'{BENCH}/algorithms/lacam/**/liblacam.so', recursive=True)
print('built:', so)
assert so, 'liblacam.so not built — see /tmp/cmake.log'
shutil.copy(so[0], f'{BENCH}/algorithms/lacam/liblacam.so')
print('placed at algorithms/lacam/liblacam.so')

## 3. Python deps
pogema-toolbox needs **pydantic v1** and numpy>=1.21 (Kaggle's numpy/pandas/yaml are
fine as-is). On Kaggle we install into the kernel Python. On Colab the native 3.12
breaks the stack, so we use the py3.10 `follower` conda env (as in `colab_baseline`).

In [ ]:
import subprocess
if KAGGLE:
    !pip install -q pogema==1.3.0 pogema-toolbox==0.1.0 'pydantic<2'
    PYRUN = 'python'
else:
    try:
        import condacolab; condacolab.check()
    except Exception:
        !pip install -q condacolab
        import condacolab; condacolab.install()   # restarts kernel; re-run this cell after
    if 'follower' not in subprocess.run(['conda','env','list'],capture_output=True,text=True).stdout:
        !conda create -y -n follower python=3.10 >/dev/null
    !conda run -n follower pip install -q --prefer-binary pogema==1.3.0 pogema-toolbox==0.1.0 'pydantic<2' tabulate
    PYRUN = 'conda run --no-capture-output -n follower python'
print('PYRUN =', PYRUN)

## 4. Results dir + scenario map + per-instance runner

In [ ]:
import yaml
if KAGGLE:
    RESULTS = '/kaggle/working/mapf-deadlock-results'
else:
    try:
        from google.colab import drive; drive.mount('/content/gdrive')
        RESULTS = '/content/gdrive/MyDrive/mapf-deadlock-results'   # <-- adjust to your Drive path
    except Exception:
        RESULTS = f'{BASE}/mapf-deadlock-results'
os.makedirs(RESULTS, exist_ok=True)
print('RESULTS =', RESULTS)
EXP = f'{BENCH}/algorithms/experiments/99-den520d-sat'
os.makedirs(EXP, exist_ok=True)
ours = yaml.safe_load(open(f'{OURS}/env/test-maps.yaml'))
yaml.safe_dump({'den520d': ours['den520d']}, open(f'{EXP}/maps.yaml','w'))
runner = '''import sys
from pathlib import Path
import yaml
from pogema_toolbox.evaluator import evaluation
from pogema_toolbox.registry import ToolboxRegistry
from create_env import create_env_base
from pogema_toolbox.create_env import Environment
from lacam.inference import LacamInference, LacamInferenceConfig
N, seed, out, steps = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3], int(sys.argv[4])
ToolboxRegistry.register_env("Environment", create_env_base, Environment)
ToolboxRegistry.register_algorithm("LaCAM", LacamInference, LacamInferenceConfig)
ToolboxRegistry.register_maps(yaml.safe_load(open("experiments/99-den520d-sat/maps.yaml")))
cfg = {"environment": {"name":"Environment","collision_system":"soft","on_target":"restart",
        "observation_type":"POMAPF","max_episode_steps":steps,"map_name":"den520d",
        "num_agents":{"grid_search":[N]},"seed":{"grid_search":[seed]}},
       "algorithms":{"LaCAM":{"name":"LaCAM","num_process":1,"parallel_backend":"sequential"}}}
Path(out).mkdir(parents=True, exist_ok=True)
evaluation(cfg, eval_dir=Path(out))
print("DONE", out)
'''
open(f'{BENCH}/algorithms/run_lacam.py','w').write(runner)
print('wrote maps.yaml + run_lacam.py')

## 5. Smoke test — 128 agents, seed 0, 128 steps  *(gate)*

In [ ]:
import glob, json
os.chdir(f'{BENCH}/algorithms')
!{PYRUN} run_lacam.py 128 0 /tmp/lacam_smoke 128
j = glob.glob('/tmp/lacam_smoke/*.json')
print('files:', j)
assert j, 'no result — check the error above before the sweep'
print('smoke throughput:', json.load(open(j[0]))[0]['metrics'].get('avg_throughput'))

## 6. Full sweep — den520d 128–640 × 3 seeds  *(resumable, persisted)*

In [ ]:
import subprocess, time
RES = f'{RESULTS}/lacam-sat'
os.chdir(f'{BENCH}/algorithms')
for N in [128, 256, 384, 512, 640]:
    for k in [0, 1, 2]:
        out = f'{RES}/n{N}_s{k}'
        if os.path.exists(f'{out}/LaCAM.json'):
            print(f'n{N} s{k}: done, skip', flush=True); continue
        print(f'=== n{N} s{k} ===', flush=True); t0 = time.time()
        r = subprocess.run(PYRUN.split() + ['run_lacam.py', str(N), str(k), out, '512'])
        ok = os.path.exists(f'{out}/LaCAM.json')
        print(f"    {'OK' if ok else f'FAIL rc={r.returncode}'} in {(time.time()-t0)/60:.1f} min", flush=True)
print('done — run results cell.')

## 7. LaCAM throughput vs Follower

In [ ]:
import glob, json
from collections import defaultdict
agg = defaultdict(list)
for fp in glob.glob(f'{RESULTS}/lacam-sat/n*_s*/LaCAM.json'):
    for r in json.load(open(fp)):
        agg[r['env_grid_search']['num_agents']].append(r['metrics']['avg_throughput'])
ref = {128:2.23, 256:2.32, 384:1.92, 512:1.56, 640:1.06}
print(f"{'agents':>6}{'LaCAM':>9}{'n':>3}{'Follower':>10}")
for n in sorted(agg):
    x = agg[n]; print(f"{n:>6}{sum(x)/len(x):>9.3f}{len(x):>3}{ref.get(n,0):>10.2f}")